# Coding Exercise 3 – Time-Series Processes
## MA(q) · AR(p) via Yule-Walker · ARMA Box-Jenkins · VAR(p)


## Imports & Global Settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import toeplitz
from statsmodels.tsa.stattools import adfuller, pacf as sm_pacf, acf as sm_acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings, os
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

T        = 10_000   # time-series length  (change freely)
EPS_MEAN = 0.5      # noise mean
EPS_VAR  = 2.0      # noise variance

# ── Load bjm.csv ──────────────────────────────────────────────────────────────
_candidates = [
    'bjm.csv',
    '/mnt/user-data/uploads/bjm_cbf3196e-2dfb-4af0-a1ac-e774f6dc7d71.csv',
]
CSV_PATH = next((p for p in _candidates if os.path.exists(p)), None)
assert CSV_PATH, "bjm.csv not found – place it in the same directory or update CSV_PATH."

bjm  = np.loadtxt(CSV_PATH, delimiter=',')
col1, col2, col3 = bjm[:, 0], bjm[:, 1], bjm[:, 2]
print(f"bjm.csv  shape={bjm.shape}")
for i, c in enumerate([col1, col2, col3], 1):
    print(f"  Col{i}: mean={c.mean():.3f}  var={c.var():.3f}")


## Shared Helper Functions

In [ ]:
def make_noise(size, eps_mean=EPS_MEAN, eps_var=EPS_VAR):
    """
    i.i.d. noise with exact mean=eps_mean, variance=eps_var.
    Uses a Gamma distribution (swap with any distribution you prefer).
    """
    k_g     = eps_mean**2 / eps_var
    theta_g = eps_var    / eps_mean
    return np.random.gamma(k_g, theta_g, size)


def emp_acov(X, max_lag):
    """Empirical autocovariances gamma_hat(0)..gamma_hat(max_lag)."""
    n, mu = len(X), X.mean()
    return np.array([np.mean((X[:n-k] - mu) * (X[k:] - mu))
                     for k in range(max_lag + 1)])


def plot_acf_pacf(series, title='', nlags=25):
    """Side-by-side ACF and PACF with 95% confidence bands."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 3))
    thr = 1.96 / np.sqrt(len(series))
    for ax, vals, lbl in zip(axes,
            [sm_acf(series, nlags=nlags, fft=True),
             sm_pacf(series, nlags=nlags)],
            ['ACF', 'PACF']):
        ax.bar(range(len(vals)), vals, alpha=0.7)
        ax.axhline( thr, color='r', ls='--', lw=1, label='95% CI')
        ax.axhline(-thr, color='r', ls='--', lw=1)
        ax.set_title(f'{lbl} – {title}')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


---
## Q1 – MA(q) Process: Generation & Verification

### Model
$$X_t = \varepsilon_t + a_1\varepsilon_{t-1} + \cdots + a_q\varepsilon_{t-q}$$

### Theoretical moments  (with $a_0 = 1$)

| Quantity | Formula |
|---|---|
| Mean | $\mu_X = \bigl(\sum_{j=0}^{q} a_j\bigr)\,\mu_\varepsilon$ |
| $\gamma(0)$ | $\sigma^2_\varepsilon \sum_{j=0}^{q} a_j^2$ |
| $\gamma(k),\;1 \le k \le q$ | $\sigma^2_\varepsilon \sum_{j=0}^{q-k} a_j\,a_{j+k}$ |
| $\gamma(k),\;k > q$ | $0$ |


In [ ]:
def generate_ma(q, a_coeffs, T=T, eps_mean=EPS_MEAN, eps_var=EPS_VAR):
    """
    MA(q): X_t = eps_t + a1*eps_{t-1} + ... + aq*eps_{t-q}.
    Works for any q and any list of q coefficients.
    """
    eps = make_noise(T + q, eps_mean, eps_var)
    a   = np.array([1.0] + list(a_coeffs))   # a[0]=1
    X   = np.zeros(T)
    for t in range(T):
        # eps[t+q], eps[t+q-1], ..., eps[t]
        X[t] = a @ eps[t: t + q + 1][::-1]
    return X, eps[q:]


def ma_theory(q, a_coeffs, eps_mean=EPS_MEAN, eps_var=EPS_VAR):
    """Theoretical mean and autocovariances gamma(0)..gamma(q+2)."""
    a   = np.array([1.0] + list(a_coeffs))
    mu  = a.sum() * eps_mean
    acv = np.zeros(q + 3)
    for k in range(q + 1):
        acv[k] = eps_var * np.dot(a[:q - k + 1], a[k:])
    return mu, acv


# ── Parameters (change freely) ────────────────────────────────────────────────
q_ma = 3
a_ma = [0.6, -0.4, 0.3]

X_ma, eps_ma   = generate_ma(q_ma, a_ma)
mu_th, acv_th  = ma_theory(q_ma, a_ma)
acv_emp        = emp_acov(X_ma, q_ma + 2)

# ── Verification table ────────────────────────────────────────────────────────
print(f"MA({q_ma})   a={a_ma}   noise: mean={EPS_MEAN}, var={EPS_VAR}")
print(f"\n{'':28}  {'Theory':>10}  {'Empirical':>10}")
print(f"{'Mean':28}  {mu_th:>10.4f}  {X_ma.mean():>10.4f}")
for k in range(q_ma + 3):
    print(f"{'gamma('+str(k)+')':28}  {acv_th[k]:>10.4f}  {acv_emp[k]:>10.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(X_ma[:300], lw=0.7)
axes[0].set_title(f'MA({q_ma}) realisation (first 300 pts)')
lags = np.arange(q_ma + 3)
axes[1].bar(lags, acv_emp, alpha=0.6, label='empirical')
axes[1].plot(lags, acv_th, 'r-o', ms=6, label='theory')
axes[1].set_title('Autocovariance'); axes[1].legend()
axes[2].hist(X_ma, bins=50, density=True, alpha=0.6)
axes[2].set_title('Histogram of X_t')
plt.tight_layout(); plt.show()

plot_acf_pacf(X_ma, title=f'MA({q_ma})')


---
## Q2 – AR(3) Process + Yule-Walker Verification

### Model
$$X_t = a_1 X_{t-1} + a_2 X_{t-2} + a_3 X_{t-3} + \varepsilon_t$$

### Theoretical mean
$$\mu_X = \frac{\mu_\varepsilon}{1 - a_1 - a_2 - a_3}$$

### Yule-Walker system
For AR($p$), the autocovariances satisfy for $k \ge 1$:
$$\gamma(k) = \sum_{j=1}^{p} a_j\,\gamma(|k-j|)$$
and at $k=0$:
$$\gamma(0) = \sum_{j=1}^{p} a_j\,\gamma(j) + \sigma^2_\varepsilon$$

These form a $(p+1)\times(p+1)$ linear system for $\gamma(0)\ldots\gamma(p)$;
higher lags follow by recursion.


In [ ]:
def generate_ar(p, a_coeffs, T=T, eps_mean=EPS_MEAN, eps_var=EPS_VAR, burn=500):
    """
    AR(p): X_t = a1*X_{t-1} + ... + ap*X_{t-p} + eps_t.
    burn-in discards initial transients so the series starts near stationarity.
    Works for any p and any list of p coefficients.
    """
    a   = np.asarray(a_coeffs)
    eps = make_noise(T + burn, eps_mean, eps_var)
    X   = np.zeros(T + burn)
    for t in range(p, T + burn):
        X[t] = a @ X[t - p: t][::-1] + eps[t]
    return X[burn:], eps[burn:]


def ar_theory_acov(p, a_coeffs, eps_var, max_lag=6):
    """
    Theoretical autocovariances gamma(0)..gamma(max_lag) via Yule-Walker.
    Solves a (p+1)x(p+1) linear system, then extends by recursion.
    """
    a    = np.asarray(a_coeffs)
    size = p + 1
    A_mat = np.eye(size)
    b_vec = np.zeros(size)
    b_vec[0] = eps_var
    for k in range(size):
        for j in range(1, p + 1):
            idx = abs(k - j)
            if idx < size:
                A_mat[k, idx] -= a[j - 1]
    gamma = list(np.linalg.solve(A_mat, b_vec))
    for k in range(p + 1, max_lag + 1):
        gamma.append(sum(a[j-1] * gamma[k-j] for j in range(1, p+1)))
    return np.array(gamma)


def ar_theory_mean(a_coeffs, eps_mean=EPS_MEAN):
    return eps_mean / (1.0 - sum(a_coeffs))


# ── Parameters (change freely; ensure stationarity) ───────────────────────────
a_ar3 = [0.5, 0.2, -0.1]

X_ar3, eps_ar3  = generate_ar(3, a_ar3)
mu_ar3_th       = ar_theory_mean(a_ar3)
acv_ar3_th      = ar_theory_acov(3, a_ar3, EPS_VAR, max_lag=5)
acv_ar3_emp     = emp_acov(X_ar3, 5)

print(f"AR(3)   a={a_ar3}   noise: mean={EPS_MEAN}, var={EPS_VAR}")
print(f"\n{'':28}  {'Theory (YW)':>12}  {'Empirical':>12}")
print(f"{'Mean':28}  {mu_ar3_th:>12.4f}  {X_ar3.mean():>12.4f}")
for k in range(6):
    print(f"{'gamma('+str(k)+')':28}  {acv_ar3_th[k]:>12.4f}  {acv_ar3_emp[k]:>12.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(X_ar3[:300], lw=0.7)
axes[0].set_title('AR(3) realisation (first 300 pts)')
lags3 = np.arange(6)
axes[1].bar(lags3, acv_ar3_emp, alpha=0.6, label='empirical')
axes[1].plot(lags3, acv_ar3_th, 'r-o', ms=6, label='YW theory')
axes[1].set_title('Autocovariance (Yule-Walker)'); axes[1].legend()
pk3 = sm_pacf(X_ar3, nlags=12)
axes[2].bar(range(len(pk3)), pk3, alpha=0.7)
axes[2].axhline( 1.96/np.sqrt(T), color='r', ls='--', label='95% CI')
axes[2].axhline(-1.96/np.sqrt(T), color='r', ls='--')
axes[2].set_title('PACF (should cut off at lag 3)'); axes[2].legend()
plt.tight_layout(); plt.show()


---
## Q3 – Identify AR(p) from `bjm.csv` Column 1

### Procedure
1. **ADF test** — confirm stationarity.  
2. **PACF plot** — AR(p) PACF cuts off sharply after lag $p$.  
3. **Yule-Walker estimation** — solve $\Gamma_p\hat{\mathbf{a}} = \hat{\boldsymbol{\gamma}}_{1:p}$ using the empirical Toeplitz matrix.  
4. **Recover noise statistics**:
$$\hat{\mu}_\varepsilon = \bar{X}\Bigl(1 - \sum_j \hat{a}_j\Bigr), \qquad
  \hat{\sigma}^2_\varepsilon = \hat{\gamma}(0) - \hat{\mathbf{a}}^\top\hat{\boldsymbol{\gamma}}_{1:p}$$
5. **Verify** — compare $\gamma(0)\ldots\gamma(5)$ from YW theory vs empirical.


In [ ]:
def yule_walker_fit(X, p):
    """
    Fit AR(p) via Yule-Walker.
    Returns: a_hat (p,), sigma2_eps (scalar), gamma_array gamma(0..p)
    """
    mu    = X.mean()
    Xc    = X - mu
    gamma = emp_acov(Xc, p)                  # gamma(0)..gamma(p)
    Gamma_p  = toeplitz(gamma[:p])           # p×p symmetric Toeplitz
    gamma_1p = gamma[1: p + 1]
    a_hat    = np.linalg.solve(Gamma_p, gamma_1p)
    sigma2   = gamma[0] - a_hat @ gamma_1p
    return a_hat, sigma2, gamma


# ── 1. Stationarity ───────────────────────────────────────────────────────────
adf_c1 = adfuller(col1)
print(f"ADF: stat={adf_c1[0]:.3f},  p={adf_c1[1]:.4f}  →  "
      f"{'Stationary ✓' if adf_c1[1] < 0.05 else 'Non-stationary – consider differencing'}")

# ── 2. ACF / PACF ─────────────────────────────────────────────────────────────
plot_acf_pacf(col1, title='bjm.csv  Col 1', nlags=20)

# ── 3. Determine p ────────────────────────────────────────────────────────────
max_lags_q3  = 15
pk_c1        = sm_pacf(col1, nlags=max_lags_q3)
thresh_c1    = 1.96 / np.sqrt(len(col1))
sig_pacf_c1  = np.where(np.abs(pk_c1[1:]) > thresh_c1)[0] + 1   # skip lag-0

print(f"Significant PACF lags: {sig_pacf_c1}")
p_hat_c1 = int(sig_pacf_c1.max()) if len(sig_pacf_c1) else 1
print(f"Chosen p = {p_hat_c1}  (last significant PACF lag)")

# ── 4. Yule-Walker fit ────────────────────────────────────────────────────────
a_hat_c1, sig2_eps_c1, gamma_c1 = yule_walker_fit(col1, p_hat_c1)

mu_eps_c1  = col1.mean() * (1.0 - a_hat_c1.sum())
var_eps_c1 = sig2_eps_c1

print(f"\n=== AR({p_hat_c1}) – bjm.csv Column 1 ===")
print(f"  Process mean         : {col1.mean():.4f}")
print(f"  AR coefficients â    : {np.round(a_hat_c1, 4)}")
print(f"  Noise mean  mu_eps   : {mu_eps_c1:.4f}")
print(f"  Noise var   sig2_eps : {var_eps_c1:.4f}")

# ── 5. Autocovariance verification ───────────────────────────────────────────
acv_th_c1  = ar_theory_acov(p_hat_c1, a_hat_c1, var_eps_c1, max_lag=5)
acv_emp_c1 = emp_acov(col1 - col1.mean(), 5)

print(f"\n  {'Lag':>4}  {'Theory (YW)':>14}  {'Empirical':>14}  {'|Rel. err|':>12}")
for k in range(6):
    rel = abs(acv_th_c1[k] - acv_emp_c1[k]) / (abs(acv_emp_c1[k]) + 1e-10)
    print(f"  {k:>4}  {acv_th_c1[k]:>14.4f}  {acv_emp_c1[k]:>14.4f}  {rel:>12.5f}")

# ── Plot fit ──────────────────────────────────────────────────────────────────
lags_q3 = np.arange(6)
plt.figure(figsize=(7, 4))
plt.bar(lags_q3, acv_emp_c1, alpha=0.6, label='empirical')
plt.plot(lags_q3, acv_th_c1, 'r-o', ms=6, label='YW theory')
plt.title(f'AR({p_hat_c1}) autocovariance – Col 1'); plt.legend(); plt.show()


---
## Q4 – Box-Jenkins ARMA Identification for `bjm.csv` Columns 2 & 3

### Box-Jenkins Steps

| Step | Action |
|------|--------|
| **1. Identification** | ADF stationarity test; ACF/PACF plots to guess $(p,q)$ |
| **2. Estimation** | AIC grid search on a fast subsample; final MLE on full data |
| **3. Diagnostics** | Ljung-Box test + residual ACF to confirm white-noise residuals |

### Reading ACF/PACF patterns
| ACF | PACF | Suggested model |
|-----|------|----------------|
| Cuts off at $q$ | Decays | MA($q$) |
| Decays | Cuts off at $p$ | AR($p$) |
| Both decay | Both decay | ARMA($p,q$) → use AIC |

> **Speed note:** ARMA MLE on 10 000 points is slow. The AIC grid runs on
> `sub_size` points (fast); only the chosen $(p,q)$ is fit on the full series.


In [ ]:
def box_jenkins(series, name, p_max=5, q_max=4, sub_size=1000):
    """
    Full Box-Jenkins pipeline for a stationary univariate series.

    Parameters
    ----------
    series   : 1-D array
    name     : label for plots and printed output
    p_max    : maximum AR order to search
    q_max    : maximum MA order to search
    sub_size : observations used for the AIC grid (speed trade-off)

    Returns
    -------
    fitted statsmodels ARIMA model, best (p, q) tuple
    """
    T_s = len(series)
    print(f"\n{'='*62}")
    print(f"  Box-Jenkins: {name}   (n={T_s:,})")
    print(f"{'='*62}")

    # ── Step 1: Stationarity ─────────────────────────────────────────────────
    adf = adfuller(series)
    print(f"  ADF: stat={adf[0]:.3f}, p={adf[1]:.4f}  → "
          f"{'Stationary ✓' if adf[1] < 0.05 else 'Non-stationary'}")

    # ── Step 2: ACF / PACF ───────────────────────────────────────────────────
    plot_acf_pacf(series, title=name)

    # ── Step 3: AIC grid on subsample ────────────────────────────────────────
    sub      = series[:sub_size]
    best_aic = np.inf
    best_ord = (1, 0)
    aic_grid = np.full((p_max + 1, q_max + 1), np.nan)

    for p in range(p_max + 1):
        for q in range(q_max + 1):
            if p + q == 0:
                continue
            try:
                m = ARIMA(sub, order=(p, 0, q)).fit()
                aic_grid[p, q] = m.aic
                if m.aic < best_aic:
                    best_aic, best_ord = m.aic, (p, q)
            except Exception:
                pass

    print(f"\n  AIC grid on first {sub_size} observations  (rows=p, cols=q):")
    print("    q=  " + "  ".join(f"{q:>8}" for q in range(q_max + 1)))
    for p in range(p_max + 1):
        row = "  ".join(
            f"{aic_grid[p, q]:>8.1f}" if not np.isnan(aic_grid[p, q]) else "       —"
            for q in range(q_max + 1))
        print(f"  p={p} {row}")

    p_best, q_best = best_ord
    print(f"\n  Best order: ARMA({p_best}, {q_best})   (subsample AIC = {best_aic:.2f})")

    # ── Step 4: Final MLE fit on full series ──────────────────────────────────
    model = ARIMA(series, order=(p_best, 0, q_best)).fit()
    print(f"\n  Full-data ARMA({p_best},{q_best})  AIC = {model.aic:.2f}")
    if p_best:
        print(f"    AR params : {np.round(model.arparams, 4)}")
    if q_best:
        print(f"    MA params : {np.round(model.maparams, 4)}")

    # Noise statistics
    ar_sum    = model.arparams.sum() if p_best else 0.0
    mu_noise  = series.mean() * (1.0 - ar_sum)
    var_noise = model.resid.var()
    print(f"    Noise mean  mu_eps  : {mu_noise:.4f}")
    print(f"    Noise var  sig2_eps : {var_noise:.4f}")

    # ── Step 5: Diagnostics ───────────────────────────────────────────────────
    resid = model.resid
    lb    = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
    lbp   = lb['lb_pvalue'].values
    print(f"\n  Ljung-Box p-values  lag=10: {lbp[0]:.4f},  lag=20: {lbp[1]:.4f}")
    print(f"  {'Residuals ~ white noise ✓' if all(lbp > 0.05) else 'Autocorrelation remains – try a higher order'}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    axes[0].plot(resid[:500], lw=0.6)
    axes[0].set_title(f'Residuals – {name}  (first 500)')
    racf = sm_acf(resid, nlags=25, fft=True)
    thr  = 1.96 / np.sqrt(T_s)
    axes[1].bar(range(len(racf)), racf, alpha=0.7)
    axes[1].axhline( thr, color='r', ls='--')
    axes[1].axhline(-thr, color='r', ls='--')
    axes[1].set_title('Residual ACF (should be flat)')
    axes[2].hist(resid, bins=50, density=True, alpha=0.6)
    xs = np.linspace(resid.min(), resid.max(), 200)
    axes[2].plot(xs, stats.norm.pdf(xs, resid.mean(), resid.std()), 'r', lw=2)
    axes[2].set_title('Residual histogram')
    plt.suptitle(f'Diagnostics: ARMA({p_best},{q_best}) – {name}', y=1.01)
    plt.tight_layout(); plt.show()

    return model, best_ord


model_c2, order_c2 = box_jenkins(col2, 'bjm.csv  Col 2')
model_c3, order_c3 = box_jenkins(col3, 'bjm.csv  Col 3')


---
## Q5 – Vector Autoregressive Process VAR(p)

### (a) Model
$$\mathbf{X}_t = A_1\mathbf{X}_{t-1} + \cdots + A_p\mathbf{X}_{t-p} + \boldsymbol{\varepsilon}_t$$

Noise: $\boldsymbol{\varepsilon}_t \stackrel{\text{iid}}{\sim}$ joint distribution with
$\boldsymbol{\mu}_\varepsilon = [0.5,\,-0.5]^\top$,
$\Sigma_\varepsilon = \bigl[\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\bigr]$

### (b) Identification
1. Fit by **OLS** for each candidate $p$:
   $\hat{B} = (Z^\top Z)^{-1} Z^\top Y$,
   where $Z$ stacks $p$ lags and $Y$ is the response matrix.
2. Compute **AIC** and **BIC** from the multivariate Gaussian log-likelihood.
3. Pick $\hat{p} = \arg\min_p \text{AIC}$ (or BIC).
4. Recover noise mean: $\hat{\boldsymbol{\mu}}_\varepsilon = (I - \sum_k \hat{A}_k)\,\bar{\mathbf{X}}$


In [ ]:
# ── (a) Generator ─────────────────────────────────────────────────────────────
def generate_var(p, A_list, T=T, mu_eps=None, Sigma_eps=None, burn=500):
    """
    VAR(p) of dimension d = A_list[0].shape[0].

    A_list : [A1, A2, ..., Ap] each (d, d).
    Works for any p, d, and noise distribution specified by mu_eps / Sigma_eps.
    """
    d = A_list[0].shape[0]
    if mu_eps    is None: mu_eps    = np.zeros(d)
    if Sigma_eps is None: Sigma_eps = np.eye(d)

    Ttot = T + burn
    eps  = np.random.multivariate_normal(mu_eps, Sigma_eps, Ttot)
    X    = np.zeros((Ttot, d))
    for t in range(p, Ttot):
        X[t] = sum(A_list[k] @ X[t - k - 1] for k in range(p)) + eps[t]
    return X[burn:], eps[burn:]


# ── (b) OLS estimator + information criteria ──────────────────────────────────
def fit_var_ols(X, p):
    """
    Fit VAR(p) by OLS (equation-by-equation).

    Returns
    -------
    A_list  : [A1_hat, ..., Ap_hat], each (d, d)
    E       : (T-p, d) residuals
    ll      : multivariate Gaussian log-likelihood
    Sigma_e : (d, d) residual covariance
    """
    T_fit, d = X.shape
    # Stacked lagged regressor matrix Z: shape (T-p, p*d)
    Z = np.column_stack([X[p - k - 1: T_fit - k - 1] for k in range(p)])
    Y = X[p:]
    B     = np.linalg.lstsq(Z, Y, rcond=None)[0]   # (p*d, d)
    E     = Y - Z @ B
    A_hat = [B[k * d: (k + 1) * d].T for k in range(p)]

    Sigma_e       = (E.T @ E) / len(E)
    _, log_det    = np.linalg.slogdet(Sigma_e)
    T_eff = len(E)
    ll    = -0.5 * T_eff * (d * np.log(2 * np.pi) + log_det + d)
    return A_hat, E, ll, Sigma_e


def var_order_select(X, p_max=8):
    """AIC and BIC for VAR(p), p = 1 .. p_max."""
    T_fit, d = X.shape
    aic_v, bic_v = [], []
    for p in range(1, p_max + 1):
        _, _, ll, _ = fit_var_ols(X, p)
        k     = p * d * d
        T_eff = T_fit - p
        aic_v.append(-2 * ll + 2 * k)
        bic_v.append(-2 * ll + k * np.log(T_eff))
    return np.array(aic_v), np.array(bic_v)


# ─────────────────────────────────────────────────────────────────────────────
# True parameters – change freely
# Check stationarity: all eigenvalues of the companion matrix must be < 1.
# ─────────────────────────────────────────────────────────────────────────────
d_var  = 2
p_true = 2

A1_true = np.array([[ 0.40,  0.10],
                     [ 0.20,  0.30]])
A2_true = np.array([[-0.10,  0.05],
                     [ 0.05, -0.10]])

mu_eps_var    = np.array([0.5, -0.5])
Sigma_eps_var = np.array([[2.0, 1.0],
                           [1.0, 2.0]])

# ── Companion eigenvalue check ────────────────────────────────────────────────
companion = np.block([
    [A1_true, A2_true],
    [np.eye(d_var), np.zeros((d_var, d_var))]
])
eigs = np.abs(np.linalg.eigvals(companion))
print(f"Companion eigenvalue moduli: {np.round(eigs, 4)}")
print("Stationarity:", "✓  all < 1" if (eigs < 1).all() else "✗  VIOLATED – adjust coefficients")

# ── (a) Generate ──────────────────────────────────────────────────────────────
X_var, eps_var_data = generate_var(
    p_true, [A1_true, A2_true],
    mu_eps=mu_eps_var, Sigma_eps=Sigma_eps_var
)
print(f"\nGenerated VAR({p_true}):  shape={X_var.shape}")
print(f"  Empirical process mean : {X_var.mean(axis=0).round(4)}")
print(f"  Empirical process cov  :\n{np.round(np.cov(X_var.T), 4)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, ax in enumerate(axes):
    ax.plot(X_var[:400, i], lw=0.7)
    ax.set_title(f'VAR({p_true}) – Component {i+1} (first 400 pts)')
plt.tight_layout(); plt.show()

# ── (b) Order selection ───────────────────────────────────────────────────────
p_max_s = 8
aic_v, bic_v = var_order_select(X_var, p_max=p_max_s)
p_aic = int(np.argmin(aic_v)) + 1
p_bic = int(np.argmin(bic_v)) + 1

print(f"\nVAR order selection  (true p = {p_true}):")
print(f"  {'p':>3}  {'AIC':>14}  {'BIC':>14}")
for p in range(1, p_max_s + 1):
    note = ("  ← AIC min" if p == p_aic else "") + ("  ← BIC min" if p == p_bic else "")
    print(f"  {p:>3}  {aic_v[p-1]:>14.2f}  {bic_v[p-1]:>14.2f}{note}")

fig, ax = plt.subplots(figsize=(8, 4))
ps = range(1, p_max_s + 1)
ax.plot(ps, aic_v, 'b-o', label='AIC')
ax.plot(ps, bic_v, 'r-s', label='BIC')
ax.axvline(p_aic, color='b', ls='--', alpha=0.5, label=f'AIC min p={p_aic}')
ax.axvline(p_bic, color='r', ls='--', alpha=0.5, label=f'BIC min p={p_bic}')
ax.set_xticks(list(ps)); ax.legend()
ax.set_title('VAR(p) order selection via AIC / BIC')
ax.set_xlabel('p'); plt.tight_layout(); plt.show()

# ── Fit at selected order ─────────────────────────────────────────────────────
p_sel = p_aic     # swap to p_bic if you prefer BIC
A_hat_list, E_hat, _, Sigma_hat = fit_var_ols(X_var, p_sel)
true_list = [A1_true, A2_true]

print(f"\n=== VAR({p_sel}) – Estimated vs True ===")
for k in range(min(p_sel, len(true_list))):
    print(f"\n  A{k+1} estimated:\n{np.round(A_hat_list[k], 4)}")
    print(f"  A{k+1} true      :\n{true_list[k]}")

# Noise mean recovery: mu_eps = (I - sum(Ak)) * mu_X
mu_eps_est = (np.eye(d_var) - sum(A_hat_list)) @ X_var.mean(axis=0)
print(f"\n  Noise mean (estimated) : {np.round(mu_eps_est, 4)}")
print(f"  Noise mean (true)      : {mu_eps_var}")
print(f"\n  Noise cov (estimated):\n{np.round(Sigma_hat, 4)}")
print(f"  Noise cov (true)      :\n{Sigma_eps_var}")

# ── Residual diagnostics ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, ax in enumerate(axes):
    ax.plot(E_hat[:400, i], lw=0.6)
    ax.set_title(f'VAR({p_sel}) residuals – Component {i+1}')
plt.tight_layout(); plt.show()


---
## Summary

| Q | Data | Method | Key result |
|---|------|--------|------------|
| 1 | Simulated | Direct formula + loop | Mean & γ(0..q+2) verified |
| 2 | Simulated | AR recursion + Yule-Walker linear system | γ(0..5) verified |
| 3 | bjm col 1 | PACF cutoff → Yule-Walker estimation | Order p, noise μ & σ² |
| 4 | bjm col 2,3 | Full Box-Jenkins pipeline | ARMA(p,q), diagnostics, noise stats |
| 5 | Simulated | Multivariate OLS + AIC/BIC | VAR order, all Ak matrices |

**Generic design:** parameter blocks are clearly labelled.
Changing any order, coefficient, or noise parameter requires only editing those blocks.
